# RNA QUBO execution notebook

This notebook generates and selects RNA sequences, builds the QUBOs, runs Aer optimization and sampling, optionally runs IBM hardware jobs, evaluates retained candidates, and saves restartable raw outputs. Summaries and figures are produced in the separate analysis notebook.


## 1. Colab setup

Setup cells for Google Colab.


In [ ]:
%pip install -q qiskit-ibm-runtime ViennaRNA qiskit-optimization qiskit-aer

In [ ]:
from google.colab import userdata
from qiskit_ibm_runtime import QiskitRuntimeService

TOKEN = userdata.get("IBM_RTP_TOKEN")

if TOKEN:
    QiskitRuntimeService.save_account(
        channel="ibm_quantum_platform",
        token=TOKEN,
        overwrite=True,
        set_as_default=True,
    )
    print("IBM Quantum account saved.")
else:
    print("No IBM_RTP_TOKEN found.")

IBM Quantum account saved.


## 2. Imports, paths, and execution settings


In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import display
from time import perf_counter

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from data import (
    add_vienna_references,
    append_synthetic_sequences,
    load_beacon_sequences,
    save_processed_sequences,
)
from model import build_conflict_lists, build_sequence_qubo, enumerate_candidate_stems
from quantum import get_ibm_backend, sample_qaoa_parameters, solve_exact, solve_qaoa_aer
from analysis import evaluate_solver_result

print(f"Project root: {PROJECT_ROOT}")
print("Project modules imported successfully.")

In [ ]:
BEACON_PATH = PROJECT_ROOT / "data" / "raw" / "beacon.csv"
RESULTS_DIR = PROJECT_ROOT / "data" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SELECTED_SEQUENCES_PATH = RESULTS_DIR / "selected_sequences.csv"
MODEL_COMPLEXITY_PATH = RESULTS_DIR / "model_complexity.csv"
DETAILED_RESULTS_PATH = RESULTS_DIR / "detailed_results.csv"
RUN_METADATA_PATH = RESULTS_DIR / "run_metadata.csv"
FAILURES_PATH = RESULTS_DIR / "failures.csv"

BEACON_MAX_SEQUENCES = 10000

SIMULATION_LENGTHS = [10, 12, 14, 16, 18, 19, 20]
SIMULATION_SEQUENCES_PER_LENGTH = 30
SIMULATION_MIN_VARIABLES = 1
SIMULATION_MAX_VARIABLES = None

RUN_HARDWARE = True
IBM_BACKEND_NAME = None
HARDWARE_LENGTHS = [10, 15, 20, 25, 30, 35, 37, 40, 41, 42, 43, 44]
HARDWARE_SEQUENCES_PER_LENGTH = 15
HARDWARE_MIN_VARIABLES = 1
HARDWARE_MAX_VARIABLES = 155
HARDWARE_SHOTS = 2048

SYNTHETIC_LENGTHS = sorted(
    set(SIMULATION_LENGTHS + (HARDWARE_LENGTHS if RUN_HARDWARE else []))
)
SYNTHETIC_PER_LENGTH = 50
SYNTHETIC_SEED = 666

MIN_STEM_LENGTH = 2
MIN_LOOP_LENGTH = 3
VARIANTS = ["strict", "relaxed", "postprocessed"]

AER_RUN_SEEDS = [666]
QAOA_REPS = 1
QAOA_OPTIMIZATION_SHOTS = 1024
QAOA_MAXITER = 25
FIXED_SAMPLING_SHOTS = 2048
MAX_SAMPLES_TO_EVALUATE = 5000
HARDWARE_PARAMETER_SEED = AER_RUN_SEEDS[0]

RUN_EXACT_SANITY = True
EXACT_MAX_VARIABLES = 14
RELOAD_AER_CHECKPOINT = False

print(f"Synthetic lengths: {SYNTHETIC_LENGTHS}")
print(f"Aer lengths: {SIMULATION_LENGTHS}")
print(f"IBM lengths: {HARDWARE_LENGTHS if RUN_HARDWARE else 'disabled'}")
print(f"Aer run seeds: {AER_RUN_SEEDS}")

Synthetic lengths: [10, 12, 14, 15, 16, 18, 19, 20, 25, 30, 35, 37, 40, 41, 42, 43, 44]
Aer lengths: [10, 12, 14, 16, 18, 19, 20]
IBM lengths: [10, 15, 20, 25, 30, 35, 37, 40, 41, 42, 43, 44]
Aer run seeds: [666]


## 3. Generate and select sequences

Synthetic sequences are ranked by candidate-stem count. The lowest-complexity eligible sequences are selected.


In [ ]:
beacon_df = load_beacon_sequences(
    BEACON_PATH,
    max_sequences=BEACON_MAX_SEQUENCES,
    max_length=None,
)

beacon_with_synthetic = append_synthetic_sequences(
    beacon_df,
    len_list=SYNTHETIC_LENGTHS,
    n_seq=SYNTHETIC_PER_LENGTH,
    seed=SYNTHETIC_SEED,
)

synthetic_pool_df = (
    beacon_with_synthetic[beacon_with_synthetic["source"] == "synth"]
    .copy()
    .reset_index(drop=True)
)

generated_counts_df = (
    synthetic_pool_df.groupby("length")
    .size()
    .rename("generated_sequences")
    .reset_index()
)

display(generated_counts_df)
print(f"Generated synthetic rows: {len(synthetic_pool_df)}")

Dropped 141 sequence(s) containing invalid characters.
Sequences loaded: 5529
Length statistics: min=43, max=1136, median=130


,length,generated_sequences
0,10,50
1,12,50
2,14,50
3,15,50
4,16,50
5,18,50
6,19,50
7,20,50
8,25,50
9,30,50


Generated synthetic rows: 850


In [ ]:
selection_rows = []

for _, row in synthetic_pool_df.iterrows():
    stems = enumerate_candidate_stems(
        row["sequence"],
        min_stem_length=MIN_STEM_LENGTH,
        min_loop_length=MIN_LOOP_LENGTH,
    )
    conflicts = build_conflict_lists(stems)
    selection_rows.append(
        {
            "sequence_id": row["sequence_id"],
            "sequence": row["sequence"],
            "length": int(row["length"]),
            "source": row["source"],
            "num_candidate_stems": len(stems),
            "num_overlap_conflicts": len(conflicts["overlap"]),
            "num_crossing_conflicts": len(conflicts["crossing"]),
        }
    )

selection_pool_df = pd.DataFrame(selection_rows)
preflight_summary_df = (
    selection_pool_df.groupby("length")
    .agg(
        generated_sequences=("sequence_id", "nunique"),
        min_variables=("num_candidate_stems", "min"),
        median_variables=("num_candidate_stems", "median"),
        max_variables=("num_candidate_stems", "max"),
        mean_overlap_conflicts=("num_overlap_conflicts", "mean"),
        mean_crossing_conflicts=("num_crossing_conflicts", "mean"),
    )
    .reset_index()
)

display(preflight_summary_df)

,length,generated_sequences,min_variables,median_variables,max_variables,mean_overlap_conflicts,mean_crossing_conflicts
0,10,50,0,2.0,10,2.38,1.38
1,12,50,0,4.0,11,9.94,5.76
2,14,50,0,7.0,29,32.96,23.24
3,15,50,0,10.0,28,47.26,32.02
4,16,50,0,11.5,69,107.70,80.72
5,18,50,2,16.0,39,138.10,113.60
6,19,50,4,17.0,57,171.34,143.56
7,20,50,0,16.0,65,168.82,152.78
8,25,50,9,39.0,69,550.22,570.86
9,30,50,2,73.0,636,5783.06,5474.08


In [ ]:
def select_lowest_complexity_sequences(
    pool_df,
    lengths,
    sequences_per_length,
    min_variables=1,
    max_variables=None,
    selection_name="selection",
):
    eligible_df = pool_df[
        pool_df["length"].isin(lengths)
        & (pool_df["num_candidate_stems"] >= min_variables)
    ].copy()

    if max_variables is not None:
        eligible_df = eligible_df[
            eligible_df["num_candidate_stems"] <= max_variables
        ].copy()

    selected_parts = []
    shortages = []

    for length in lengths:
        length_pool = eligible_df[eligible_df["length"] == length].sort_values(
            ["num_candidate_stems", "sequence_id"],
            kind="stable",
        )

        if len(length_pool) < sequences_per_length:
            shortages.append(
                {
                    "selection": selection_name,
                    "length": length,
                    "available": len(length_pool),
                    "required": sequences_per_length,
                }
            )
        else:
            selected_parts.append(length_pool.head(sequences_per_length))

    if shortages:
        display(pd.DataFrame(shortages))
        raise ValueError(
            f"Not enough eligible sequences for {selection_name}. "
            "Increase SYNTHETIC_PER_LENGTH, relax the variable limits, "
            "or reduce the requested sequences per length."
        )

    return (
        pd.concat(selected_parts, ignore_index=True)
        .sort_values(["length", "num_candidate_stems", "sequence_id"], kind="stable")
        .reset_index(drop=True)
    )


def summarize_selection(selection_df, selection_name):
    if selection_df.empty:
        return pd.DataFrame()

    summary_df = (
        selection_df.groupby("length")
        .agg(
            selected_sequences=("sequence_id", "nunique"),
            min_variables=("num_candidate_stems", "min"),
            mean_variables=("num_candidate_stems", "mean"),
            max_variables=("num_candidate_stems", "max"),
        )
        .reset_index()
    )
    summary_df.insert(0, "selection", selection_name)
    return summary_df


In [ ]:
simulation_selection_df = select_lowest_complexity_sequences(
    selection_pool_df,
    SIMULATION_LENGTHS,
    SIMULATION_SEQUENCES_PER_LENGTH,
    SIMULATION_MIN_VARIABLES,
    SIMULATION_MAX_VARIABLES,
    "Aer benchmark",
)

hardware_selection_df = (
    select_lowest_complexity_sequences(
        selection_pool_df,
        HARDWARE_LENGTHS,
        HARDWARE_SEQUENCES_PER_LENGTH,
        HARDWARE_MIN_VARIABLES,
        HARDWARE_MAX_VARIABLES,
        "IBM subset",
    )
    if RUN_HARDWARE
    else selection_pool_df.iloc[0:0].copy()
)

selection_summary_df = pd.concat(
    [
        summarize_selection(simulation_selection_df, "Aer benchmark"),
        summarize_selection(hardware_selection_df, "IBM subset"),
    ],
    ignore_index=True,
)

display(selection_summary_df)


,selection,length,selected_sequences,min_variables,mean_variables,max_variables
0,Aer benchmark,10,30,1,1.600000,3
1,Aer benchmark,12,30,1,3.700000,7
2,Aer benchmark,14,30,1,4.700000,8
3,Aer benchmark,16,30,1,7.566667,14
4,Aer benchmark,18,30,2,11.733333,20
5,Aer benchmark,19,30,4,11.900000,18
6,Aer benchmark,20,30,2,11.933333,18
7,IBM subset,10,15,1,1.000000,1
8,IBM subset,15,15,1,4.866667,7
9,IBM subset,20,15,2,8.133333,11


In [ ]:
simulation_sequence_ids = simulation_selection_df["sequence_id"].tolist()
hardware_sequence_ids = hardware_selection_df["sequence_id"].tolist()

all_selected_sequences_df = (
    pd.concat([simulation_selection_df, hardware_selection_df], ignore_index=True)
    .drop_duplicates(subset="sequence_id", keep="first")
    .reset_index(drop=True)
)
all_selected_sequences_df["selected_for_simulation"] = all_selected_sequences_df[
    "sequence_id"
].isin(simulation_sequence_ids)
all_selected_sequences_df["selected_for_hardware"] = all_selected_sequences_df[
    "sequence_id"
].isin(hardware_sequence_ids)
all_selected_sequences_df = add_vienna_references(all_selected_sequences_df)

selected_lookup = all_selected_sequences_df.set_index("sequence_id")
simulation_selection_df = selected_lookup.loc[simulation_sequence_ids].reset_index()
hardware_selection_df = (
    selected_lookup.loc[hardware_sequence_ids].reset_index()
    if hardware_sequence_ids
    else all_selected_sequences_df.iloc[0:0].copy()
)

save_processed_sequences(all_selected_sequences_df, SELECTED_SEQUENCES_PATH)

display(
    all_selected_sequences_df[
        [
            "sequence_id",
            "length",
            "num_candidate_stems",
            "selected_for_simulation",
            "selected_for_hardware",
            "reference_structure",
            "reference_mfe",
        ]
    ]
)


Sequences saved: 360
Output path: /content/drive/MyDrive/Colab Notebooks/WISER_Moderna_02/data/results/selected_sequences.csv


,sequence_id,length,num_candidate_stems,selected_for_simulation,selected_for_hardware,reference_structure,reference_mfe
0,synth_000008,10,1,True,True,..........,0.0
1,synth_000010,10,1,True,True,..........,0.0
2,synth_000012,10,1,True,True,..........,0.0
3,synth_000016,10,1,True,True,..........,0.0
4,synth_000021,10,1,True,True,..........,0.0
...,...,...,...,...,...,...,...
355,synth_000850,44,133,False,True,(((..(((.....(((.((.....)).)))....)))..)))..,-11.4
356,synth_000849,44,135,False,True,....((((.....))))........((((...))))........,-2.4
357,synth_000846,44,136,False,True,..(((.........((((.......))))........)))....,-2.1
358,synth_000838,44,142,False,True,.(((......)))..(((((.......))))).((((...)))),-5.0


## 4. Build the QUBOs

Build the QUBO objects for execution. A row-level complexity file is saved for the analysis notebook.


In [ ]:
qubo_models = {}
model_metadata = {}
model_complexity_rows = []

for _, sequence_row in all_selected_sequences_df.iterrows():
    sequence_id = sequence_row["sequence_id"]

    for variant in VARIANTS:
        quadratic_program, metadata = build_sequence_qubo(
            sequence_row,
            variant=variant,
            min_stem_length=MIN_STEM_LENGTH,
            min_loop_length=MIN_LOOP_LENGTH,
        )
        qubo_models[(sequence_id, variant)] = quadratic_program
        model_metadata[(sequence_id, variant)] = metadata
        model_complexity_rows.append(
            {
                "sequence_id": sequence_id,
                "length": int(sequence_row["length"]),
                "variant": variant,
                "selected_for_simulation": bool(
                    sequence_row["selected_for_simulation"]
                ),
                "selected_for_hardware": bool(
                    sequence_row["selected_for_hardware"]
                ),
                "num_variables": metadata["num_variables"],
                "num_quadratic_terms": metadata["num_quadratic_terms"],
            }
        )

model_complexity_df = pd.DataFrame(model_complexity_rows)
model_complexity_df.to_csv(MODEL_COMPLEXITY_PATH, index=False)

display(model_complexity_df.head(12))
print(f"Constructed QUBOs: {len(qubo_models)}")
print(f"Model complexity saved to {MODEL_COMPLEXITY_PATH}")


,sequence_id,length,variant,selected_for_simulation,selected_for_hardware,num_variables,num_quadratic_terms
0,synth_000008,10,strict,True,True,1,0
1,synth_000008,10,relaxed,True,True,1,0
2,synth_000008,10,postprocessed,True,True,1,0
3,synth_000010,10,strict,True,True,1,0
4,synth_000010,10,relaxed,True,True,1,0
5,synth_000010,10,postprocessed,True,True,1,0
6,synth_000012,10,strict,True,True,1,0
7,synth_000012,10,relaxed,True,True,1,0
8,synth_000012,10,postprocessed,True,True,1,0
9,synth_000016,10,strict,True,True,1,0


Constructed QUBOs: 1080
Model complexity saved to /content/drive/MyDrive/Colab Notebooks/WISER_Moderna_02/data/results/model_complexity.csv


In [ ]:
if RUN_EXACT_SANITY:
    sanity_row = simulation_selection_df.sort_values(
        ["num_candidate_stems", "length", "sequence_id"],
        kind="stable",
    ).iloc[0]
    sanity_sequence_id = sanity_row["sequence_id"]
    sanity_variable_count = int(sanity_row["num_candidate_stems"])

    if sanity_variable_count <= EXACT_MAX_VARIABLES:
        exact_rows = []

        for variant in VARIANTS:
            result = solve_exact(qubo_models[(sanity_sequence_id, variant)])
            exact_rows.append(
                {
                    "sequence_id": sanity_sequence_id,
                    "length": int(sanity_row["length"]),
                    "variant": variant,
                    "num_variables": sanity_variable_count,
                    "best_objective": result["best_fval"],
                    "runtime_seconds": result["runtime_seconds"],
                }
            )

        display(pd.DataFrame(exact_rows))
    else:
        print(
            "Exact sanity check skipped: the smallest selected QUBO has "
            f"{sanity_variable_count} variables."
        )


,sequence_id,length,variant,num_variables,best_objective,runtime_seconds
0,synth_000008,10,strict,1,-2.0,0.016261
1,synth_000008,10,relaxed,1,-2.0,0.002979
2,synth_000008,10,postprocessed,1,-2.0,0.002851


## 5. Aer optimization and fixed-parameter sampling

`RELOAD_AER_CHECKPOINT = True` reuses previously saved Aer results and optimized parameters.


In [ ]:
cell_start_time = perf_counter()

detailed_frames = []
run_metadata_rows = []
failure_rows = []
aer_parameter_store = {}

if RELOAD_AER_CHECKPOINT:
    saved_detailed_df = pd.read_csv(DETAILED_RESULTS_PATH)
    saved_run_metadata_df = pd.read_csv(RUN_METADATA_PATH)

    aer_detailed_results_df = saved_detailed_df[
        saved_detailed_df["backend_mode"] == "aer"
    ].copy()
    aer_run_metadata_df = saved_run_metadata_df[
        saved_run_metadata_df["backend_mode"] == "aer"
    ].copy()

    if aer_detailed_results_df.empty or aer_run_metadata_df.empty:
        raise RuntimeError("The saved checkpoint contains no successful Aer results.")

    detailed_frames.append(aer_detailed_results_df)
    run_metadata_rows.extend(aer_run_metadata_df.to_dict("records"))

    for _, row in aer_run_metadata_df.dropna(
        subset=["optimal_parameters_json"]
    ).iterrows():
        aer_parameter_store[
            (row["sequence_id"], row["variant"], int(row["run_seed"]))
        ] = json.loads(row["optimal_parameters_json"])

    if FAILURES_PATH.exists():
        saved_failures_df = pd.read_csv(FAILURES_PATH)
        failure_rows.extend(
            saved_failures_df[saved_failures_df["stage"] == "aer"].to_dict(
                "records"
            )
        )

    print(f"Reloaded {len(aer_run_metadata_df)} successful Aer runs.")

else:
    total_sequences = len(simulation_selection_df)

    for sequence_position, (_, sequence_row) in enumerate(
        simulation_selection_df.iterrows(),
        start=1,
    ):
        sequence_id = sequence_row["sequence_id"]
        print(
            f"[{sequence_position:02d}/{total_sequences:02d}] "
            f"{sequence_id} | length={int(sequence_row['length'])} | "
            f"variables={int(sequence_row['num_candidate_stems'])}"
        )

        for variant in VARIANTS:
            quadratic_program = qubo_models[(sequence_id, variant)]
            metadata = model_metadata[(sequence_id, variant)]

            for run_index, run_seed in enumerate(AER_RUN_SEEDS, start=1):
                run_id = f"aer_run_{run_index:02d}_seed_{run_seed}"

                try:
                    optimized_result = solve_qaoa_aer(
                        quadratic_program=quadratic_program,
                        reps=QAOA_REPS,
                        shots=QAOA_OPTIMIZATION_SHOTS,
                        maxiter=QAOA_MAXITER,
                        seed=run_seed,
                    )
                    parameters = optimized_result["optimal_parameters"]

                    if parameters is None:
                        raise ValueError("Aer QAOA returned no optimized parameters.")

                    aer_parameter_store[(sequence_id, variant, run_seed)] = parameters

                    fixed_result = sample_qaoa_parameters(
                        quadratic_program=quadratic_program,
                        parameters=parameters,
                        backend_mode="aer",
                        reps=QAOA_REPS,
                        shots=FIXED_SAMPLING_SHOTS,
                        seed=run_seed,
                    )
                    evaluated_df = evaluate_solver_result(
                        solver_result=fixed_result,
                        sequence_row=sequence_row,
                        metadata=metadata,
                        max_samples=MAX_SAMPLES_TO_EVALUATE,
                        repair=True,
                    )
                    evaluated_df["run_id"] = run_id
                    evaluated_df["run_seed"] = run_seed
                    detailed_frames.append(evaluated_df)

                    run_metadata_rows.append(
                        {
                            "sequence_id": sequence_id,
                            "length": int(sequence_row["length"]),
                            "variant": variant,
                            "backend_mode": "aer",
                            "backend_name": fixed_result["backend_name"],
                            "run_id": run_id,
                            "run_seed": run_seed,
                            "num_variables": metadata["num_variables"],
                            "num_quadratic_terms": metadata["num_quadratic_terms"],
                            "num_qubits": fixed_result["num_qubits"],
                            "circuit_depth": fixed_result["circuit_depth"],
                            "best_optimized_objective": optimized_result["best_fval"],
                            "optimizer_evaluations": optimized_result[
                                "optimizer_evaluations"
                            ],
                            "optimization_runtime_seconds": optimized_result[
                                "runtime_seconds"
                            ],
                            "sampling_runtime_seconds": fixed_result[
                                "runtime_seconds"
                            ],
                            "optimization_shots": QAOA_OPTIMIZATION_SHOTS,
                            "sampling_shots": FIXED_SAMPLING_SHOTS,
                            "qaoa_reps": QAOA_REPS,
                            "retained_samples": len(evaluated_df),
                            "optimal_parameters_json": json.dumps(parameters),
                            "job_id": None,
                        }
                    )

                except Exception as error:
                    failure_rows.append(
                        {
                            "stage": "aer",
                            "sequence_id": sequence_id,
                            "length": int(sequence_row["length"]),
                            "variant": variant,
                            "run_id": run_id,
                            "run_seed": run_seed,
                            "error_type": type(error).__name__,
                            "error_message": str(error),
                        }
                    )

    aer_frames = [
        frame for frame in detailed_frames
        if not frame.empty and frame["backend_mode"].eq("aer").all()
    ]

    if not aer_frames:
        raise RuntimeError("No successful Aer result rows are available.")

    aer_detailed_results_df = pd.concat(aer_frames, ignore_index=True)
    aer_run_metadata_df = pd.DataFrame(run_metadata_rows)

    aer_detailed_results_df.to_csv(DETAILED_RESULTS_PATH, index=False)
    aer_run_metadata_df.to_csv(RUN_METADATA_PATH, index=False)

    if failure_rows:
        pd.DataFrame(failure_rows).to_csv(FAILURES_PATH, index=False)
    else:
        FAILURES_PATH.unlink(missing_ok=True)

    print("Aer checkpoint saved.")

cell_runtime_seconds = perf_counter() - cell_start_time

print(f"Successful Aer runs: {len(aer_run_metadata_df)}")
print(f"Detailed Aer candidate rows: {len(aer_detailed_results_df)}")
print(f"Real total execution time: {cell_runtime_seconds:.2f}")

[01/210] synth_000008 | length=10 | variables=1
[02/210] synth_000010 | length=10 | variables=1
[03/210] synth_000012 | length=10 | variables=1
[04/210] synth_000016 | length=10 | variables=1
[05/210] synth_000021 | length=10 | variables=1
[06/210] synth_000024 | length=10 | variables=1
[07/210] synth_000025 | length=10 | variables=1
[08/210] synth_000028 | length=10 | variables=1
[09/210] synth_000032 | length=10 | variables=1
[10/210] synth_000035 | length=10 | variables=1
[11/210] synth_000039 | length=10 | variables=1
[12/210] synth_000042 | length=10 | variables=1
[13/210] synth_000044 | length=10 | variables=1
[14/210] synth_000047 | length=10 | variables=1
[15/210] synth_000050 | length=10 | variables=1
[16/210] synth_000004 | length=10 | variables=2
[17/210] synth_000011 | length=10 | variables=2
[18/210] synth_000017 | length=10 | variables=2
[19/210] synth_000022 | length=10 | variables=2
[20/210] synth_000023 | length=10 | variables=2
[21/210] synth_000029 | length=10 | vari

## 6. Parameter transfer and IBM sampling

A hardware target uses its own Aer-optimized parameters when available. Otherwise, it receives parameters from the successful Aer donor of the same QUBO variant with the largest variable count.


In [ ]:
ibm_backend = None

if RUN_HARDWARE:
    required_qubits = int(hardware_selection_df["num_candidate_stems"].max())
    ibm_backend = get_ibm_backend(
        min_num_qubits=required_qubits,
        backend_name=IBM_BACKEND_NAME,
    )

In [ ]:
simulation_lookup = simulation_selection_df.set_index("sequence_id")


def choose_hardware_parameters(sequence_row, variant):
    sequence_id = sequence_row["sequence_id"]
    own_key = (sequence_id, variant, HARDWARE_PARAMETER_SEED)

    if own_key in aer_parameter_store:
        return {
            "parameters": aer_parameter_store[own_key],
            "parameter_source": "own_aer_parameters",
            "donor_sequence_id": sequence_id,
            "donor_length": int(sequence_row["length"]),
            "donor_num_variables": int(sequence_row["num_candidate_stems"]),
        }

    donor_ids = [
        donor_sequence_id
        for donor_sequence_id, donor_variant, donor_seed in aer_parameter_store
        if donor_variant == variant and donor_seed == HARDWARE_PARAMETER_SEED
    ]

    if not donor_ids:
        return None

    donor_sequence_id = sorted(
        donor_ids,
        key=lambda candidate_id: (
            -int(simulation_lookup.loc[candidate_id, "num_candidate_stems"]),
            candidate_id,
        ),
    )[0]
    donor_row = simulation_lookup.loc[donor_sequence_id]

    return {
        "parameters": aer_parameter_store[
            (donor_sequence_id, variant, HARDWARE_PARAMETER_SEED)
        ],
        "parameter_source": "transferred_max_variables",
        "donor_sequence_id": donor_sequence_id,
        "donor_length": int(donor_row["length"]),
        "donor_num_variables": int(donor_row["num_candidate_stems"]),
    }


In [ ]:
cell_start_time = perf_counter()

hardware_detailed_frames = []

if RUN_HARDWARE:
    total_hardware_sequences = len(hardware_selection_df)

    for sequence_position, (_, sequence_row) in enumerate(
        hardware_selection_df.iterrows(),
        start=1,
    ):
        sequence_id = sequence_row["sequence_id"]
        print(
            f"[IBM {sequence_position:02d}/{total_hardware_sequences:02d}] "
            f"{sequence_id} | length={int(sequence_row['length'])}"
        )

        for variant in VARIANTS:
            parameter_info = choose_hardware_parameters(sequence_row, variant)
            run_id = f"ibm_seed_{HARDWARE_PARAMETER_SEED}"

            if parameter_info is None:
                failure_rows.append(
                    {
                        "stage": "hardware_parameter_preparation",
                        "sequence_id": sequence_id,
                        "length": int(sequence_row["length"]),
                        "variant": variant,
                        "run_id": run_id,
                        "run_seed": HARDWARE_PARAMETER_SEED,
                        "error_type": "NoParameterDonor",
                        "error_message": (
                            "No successful Aer donor with the same variant and seed "
                            "was available."
                        ),
                    }
                )
                continue

            try:
                quadratic_program = qubo_models[(sequence_id, variant)]
                metadata = model_metadata[(sequence_id, variant)]
                hardware_result = sample_qaoa_parameters(
                    quadratic_program=quadratic_program,
                    parameters=parameter_info["parameters"],
                    backend_mode="ibm",
                    backend=ibm_backend,
                    reps=QAOA_REPS,
                    shots=HARDWARE_SHOTS,
                    seed=HARDWARE_PARAMETER_SEED,
                )
                evaluated_df = evaluate_solver_result(
                    solver_result=hardware_result,
                    sequence_row=sequence_row,
                    metadata=metadata,
                    max_samples=MAX_SAMPLES_TO_EVALUATE,
                    repair=True,
                )
                evaluated_df["run_id"] = run_id
                evaluated_df["run_seed"] = HARDWARE_PARAMETER_SEED
                hardware_detailed_frames.append(evaluated_df)

                run_metadata_rows.append(
                    {
                        "sequence_id": sequence_id,
                        "length": int(sequence_row["length"]),
                        "variant": variant,
                        "backend_mode": "ibm",
                        "backend_name": hardware_result["backend_name"],
                        "run_id": run_id,
                        "run_seed": HARDWARE_PARAMETER_SEED,
                        "num_variables": metadata["num_variables"],
                        "num_quadratic_terms": metadata["num_quadratic_terms"],
                        "num_qubits": hardware_result["num_qubits"],
                        "circuit_depth": hardware_result["circuit_depth"],
                        "sampling_runtime_seconds": hardware_result[
                            "runtime_seconds"
                        ],
                        "sampling_shots": HARDWARE_SHOTS,
                        "qaoa_reps": QAOA_REPS,
                        "retained_samples": len(evaluated_df),
                        "parameter_source": parameter_info["parameter_source"],
                        "donor_sequence_id": parameter_info["donor_sequence_id"],
                        "donor_length": parameter_info["donor_length"],
                        "donor_num_variables": parameter_info[
                            "donor_num_variables"
                        ],
                        "job_id": hardware_result["job_id"],
                    }
                )

            except Exception as error:
                failure_rows.append(
                    {
                        "stage": "ibm",
                        "sequence_id": sequence_id,
                        "length": int(sequence_row["length"]),
                        "variant": variant,
                        "run_id": run_id,
                        "run_seed": HARDWARE_PARAMETER_SEED,
                        "error_type": type(error).__name__,
                        "error_message": str(error),
                    }
                )

hardware_detailed_results_df = (
    pd.concat(hardware_detailed_frames, ignore_index=True)
    if hardware_detailed_frames
    else pd.DataFrame()
)

if not hardware_detailed_results_df.empty:
    detailed_frames.append(hardware_detailed_results_df)

if not detailed_frames:
    raise RuntimeError("No successful Aer or IBM result rows are available.")

all_detailed_results_df = pd.concat(detailed_frames, ignore_index=True)
run_metadata_df = pd.DataFrame(run_metadata_rows)
failures_df = pd.DataFrame(failure_rows) if failure_rows else pd.DataFrame()

all_detailed_results_df.to_csv(DETAILED_RESULTS_PATH, index=False)
run_metadata_df.to_csv(RUN_METADATA_PATH, index=False)

if failures_df.empty:
    FAILURES_PATH.unlink(missing_ok=True)
else:
    failures_df.to_csv(FAILURES_PATH, index=False)

cell_runtime_seconds = perf_counter() - cell_start_time

print(f"Detailed result rows: {len(all_detailed_results_df)}")
print(f"Successful run rows: {len(run_metadata_df)}")
print(f"Recorded failures: {len(failures_df)}")

print("Saved persistent outputs:")
print(f"- {SELECTED_SEQUENCES_PATH}")
print(f"- {MODEL_COMPLEXITY_PATH}")
print(f"- {DETAILED_RESULTS_PATH}")
print(f"- {RUN_METADATA_PATH}")
if not failures_df.empty:
    print(f"- {FAILURES_PATH}")

print(f"Real total execution time: {cell_runtime_seconds:.2f}")

[IBM 01/180] synth_000008 | length=10
[IBM 02/180] synth_000010 | length=10
[IBM 03/180] synth_000012 | length=10
[IBM 04/180] synth_000016 | length=10
[IBM 05/180] synth_000021 | length=10
[IBM 06/180] synth_000024 | length=10
[IBM 07/180] synth_000025 | length=10
[IBM 08/180] synth_000028 | length=10
[IBM 09/180] synth_000032 | length=10
[IBM 10/180] synth_000035 | length=10
[IBM 11/180] synth_000039 | length=10
[IBM 12/180] synth_000042 | length=10
[IBM 13/180] synth_000044 | length=10
[IBM 14/180] synth_000047 | length=10
[IBM 15/180] synth_000050 | length=10
[IBM 16/180] synth_000180 | length=15
[IBM 17/180] synth_000200 | length=15
[IBM 18/180] synth_000171 | length=15
[IBM 19/180] synth_000183 | length=15
[IBM 20/180] synth_000151 | length=15
[IBM 21/180] synth_000156 | length=15
[IBM 22/180] synth_000159 | length=15
[IBM 23/180] synth_000174 | length=15
[IBM 24/180] synth_000195 | length=15
[IBM 25/180] synth_000188 | length=15
[IBM 26/180] synth_000190 | length=15
[IBM 27/180]

## 7. Execution coverage and failures

Operational check only.


In [ ]:
execution_coverage_df = (
    run_metadata_df.groupby(["length", "variant", "backend_mode"])
    .agg(
        successful_sequences=("sequence_id", "nunique"),
        successful_runs=("run_id", "size"),
    )
    .reset_index()
)
display(execution_coverage_df)

if failures_df.empty:
    print("No failures were recorded.")
else:
    failure_summary_df = (
        failures_df.groupby(
            ["stage", "length", "variant", "error_type"],
            dropna=False,
        )
        .size()
        .rename("failure_count")
        .reset_index()
    )
    display(failure_summary_df)


,length,variant,backend_mode,successful_sequences,successful_runs
0,10,postprocessed,aer,30,30
1,10,postprocessed,ibm,15,15
2,10,relaxed,aer,30,30
3,10,relaxed,ibm,15,15
4,10,strict,aer,30,30
5,10,strict,ibm,15,15
6,12,postprocessed,aer,30,30
7,12,relaxed,aer,30,30
8,12,strict,aer,30,30
9,14,postprocessed,aer,30,30


No failures were recorded.
